<p> <center> <a href="../start_here.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 54.5%; text-align: right;">
        <a >1</a>
        <a href="02_OpenCode_setup.ipynb">2</a>
        <a href="03_introduction_mcp.ipynb">3</a>
        <a href="04_low_level_mcp.ipynb">4</a>
        <a href="05_langraph_agent.ipynb">5</a>
        <a href="06_nemo_agent_toolkit.ipynb">6</a>
        <a href="07_challenge.ipynb">7</a>
        <a href="bonus_challenge/08_bonus_challenge.ipynb">8</a>
    </span>
    <span style="float: left; width: 45%; text-align: right;"><a href="02_OpenCode_setup.ipynb">Next Notebook</a></span>
</div>

## Learning objectives

By the end of this module, participants will be able to:
- Understand what NVIDIA Inference Microservices (NIMs) are and their role in accelerated AI inference
- Obtain and configure an NVIDIA API Key from the NVIDIA API Catalog
- Send inference requests to NIM cloud or local endpoints using Python's requests library
- Parse and interpret chat completion responses from the OpenAI-compatible NIM API

## Environment Setup

In [ ]:
import os
import socket

def get_or_allocate_port(variable_name: str) -> str:
    """Use an existing port setting, or select an available local port."""
    port = os.environ.get(variable_name)
    if port:
        return port

    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(("", 0))
        port = str(sock.getsockname()[1])

    os.environ[variable_name] = port
    print(f"{variable_name} was not set; using port {port}")
    return port

for port_variable in ("MCP_PORT", "PHOENIX_PORT", "NIM_PORT"):
    get_or_allocate_port(port_variable)

os.environ.setdefault("MODEL_ID", "nvidia/nemotron-3-nano")
host = os.environ.get("HOSTNAME", "localhost")
os.environ["INF_URL"] = f"http://{host}:{os.environ['NIM_PORT']}/v1"

## Using NVIDIA Inference Microservices (NIMs)

NIMs are quickly accessible via easy-to-use open APIs available at [NVIDIA API Catalog](https://build.nvidia.com/explore/discover), a platform for accessing a wide range of microservices online. To start with NIMs, you need an `NVIDIA API Key` which requires registration. You can register by `clicking on the login button to enter your email address`, as shown in the screenshot below, and follow the rest process or attempt to generate the API Key via the [NVIDIA NGC](https://ngc.nvidia.com/signin) registration (*click on your account name -> setup -> Generate Personal Key*). After completing the process, please save your API Key somewhere you can access for future use. A sample API Key should start with `nvapi-` and 64 other characters, including underscore `_`.

If you already have an account please follow this step to get your NVIDIA API KEY:

- Login to your account from [here](https://build.nvidia.com/explore/discover).
- Click on your model of choice.
- Under Input select the Python tab, and click Get API Key and then click Generate Key.
- Copy and save the generated key as NVIDIA_API_KEY. From there, you should have access to the endpoints.

<div style="text-align: center;">
  <img src="images/nim-catalog.png" style="width: 900px; height: auto;">
</div>


In [ ]:
import os
import getpass
import warnings
warnings.filterwarnings("ignore")
if not os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    nvapi_key = getpass.getpass("Enter your NVIDIA API key: ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
    os.environ["NGC_API_KEY"] = nvapi_key

## Getting Started

This lab can be completed using either of the following methods:

- **Cloud Endpoint** – Access NVIDIA NIMs via hosted on Cloud GPUs
- **Local Endpoint** – Deploy NIMs locally on your own GPU

## Cloud Endpoint

In [ ]:
import requests
import json
from pprint import pprint

url = "https://integrate.api.nvidia.com/v1/chat/completions"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {os.environ['NVIDIA_API_KEY']}"
}

payload = {
    "model": "nvidia/nemotron-3-nano-30b-a3b",
    "messages": [{"role": "system", "content": "What is 2+2?"}],
    "temperature": 0.6,
    "top_p": 0.95,
    "max_tokens": 4096,
    "frequency_penalty": 0,
    "presence_penalty": 0,
    "stream": False
}

response = requests.post(url, headers=headers, json=payload)

In [ ]:
pprint(response.json())

## Local Endpoint

### Self-Hosted NIMs

Please execute the cell below to ensure that your docker daemon is up and running.

In [ ]:
! docker ps 

**Expected Output (if you have no running containers):**

```python

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES

```

### Login to NVCR (NVIDIA Container Registry)

To access a NIM docker image, you must login via `docker login nvcr.io.` This process requires a default username as `--username $oauthtoken` and `--password-stdin` that accepts the value of `$NGC_API_KEY.`

In [ ]:
! echo -e "$NGC_API_KEY" | docker login nvcr.io --username '$oauthtoken' --password-stdin

**Expected Output**:
```
WARNING! Your password will be stored unencrypted in /home/$USER/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store

Login Succeeded
```

#### Setting up Cache for the Model Artifacts

The NIMs download a number of files for ensuring the best profiles are selected to achieve max performance on hardware. Set up location for caching the model artifacts as `LOCAL_NIM_CACHE` and export the variable.

In [ ]:
from os.path import expanduser
home = expanduser("~")
os.environ['LOCAL_NIM_CACHE']=f"{home}/.cache/nim"
!echo $LOCAL_NIM_CACHE

In [ ]:
!mkdir -p "$LOCAL_NIM_CACHE"
# !chmod 777 "$LOCAL_NIM_CACHE"

### Launch NIM LLM Microservice

Launch the NIM LLM microservice by executing the docker run command in the cell bellow. The runtime values for `CONTAINER_PORT` and `NIM_CONTAINER_NAME` are assigned in the next cell so each session can use its own host port and container name.

```python
docker run -d \
    --gpus all \
    --name=$NIM_CONTAINER_NAME \
    --shm-size=16GB \
    -e NGC_API_KEY=$NGC_API_KEY \
    -e NIM_PASSTHROUGH_ARGS="--enable-auto-tool-choice --tool-call-parser qwen3_coder --reasoning-parser nemotron_v3" \
    -v "$LOCAL_NIM_CACHE:/opt/nim/.cache" \
    -p 8000:8000 \
    nvcr.io/nim/nvidia/nemotron-3-nano:latest
```

This Docker command launches NIM LLM microservice using the following flags:

- `-d`: Runs the container in detached mode (in the background)
- `--gpus all`: Allows the container to access all available GPUs
- `--name=$NIM_CONTAINER_NAME`: Names the container using the current user and selected host port
- `--shm-size=16GB`: Sets the size of /dev/shm to 16GB
- `-e NGC_API_KEY`: Passes the NGC_API_KEY environment variable to the container
- `e NIM_PASSTHROUGH_ARGS="--enable-auto-tool-choice --tool-call-parser qwen3_coder --reasoning-parser nemotron_v3"`: additional args to pass to vllm
- `-v $LOCAL_NIM_CACHE:/opt/nim/.cache`: Mounts the local NIM cache directory to /opt/nim/.cache in the container
- `-p $CONTAINER_PORT:8000`: Maps a dynamically selected host port to port 8000 in the container
- `nvcr.io/nim/nvidia/nemotron-3-nano:latest`: Specifies the Docker image to use

A system can have multiple running processes, so the Environment Setup cell selects free ports when they have not already been supplied. The following code derives a unique container name for the current user session:

In [ ]:
import getpass
import os
import re

def sanitize_container_component(value: str) -> str:
    sanitized = re.sub(r"[^a-zA-Z0-9_.-]+", "-", value.lower()).strip(".-_")
    return sanitized or "user"

username = sanitize_container_component(getpass.getuser())
os.environ['NIM_CONTAINER_NAME'] = f"llm-nim-{username}-{os.environ['NIM_PORT']}"

In [ ]:
! docker run -it -d --rm \
--gpus "device=${CUDA_VISIBLE_DEVICES}" \
--name=$NIM_CONTAINER_NAME \
--shm-size=16GB  \
-e NGC_API_KEY \
-e NIM_PASSTHROUGH_ARGS="--enable-auto-tool-choice --tool-call-parser qwen3_coder --reasoning-parser nemotron_v3" \
-v $LOCAL_NIM_CACHE:/opt/nim/.cache \
-p $NIM_PORT:8000 \
nvcr.io/nim/nvidia/nemotron-3-nano:latest

In [ ]:
! docker logs -f --tail 100 $NIM_CONTAINER_NAME

### Initiate A Quick Test
You can quickly test that your NIM is up and running via two methods:
- LangChain NVIDIA Endpoints
- A simple OpenAI completion request

**Parameter description:**
- **base_url**: The ULR where the NIM docker image is deployed.
- **model**: The name of the NIM model deployed. 
- **temperature**: To modulate the randomness of sampling. Reducing the temperature increases the chance of selecting words with high probabilities.
- **top_p**: To control how deterministic the model is. If you are looking for exact and factual answers, keep this low. If you seek more diverse responses, increase to a higher value.
- **max_tokens**: maximum number of output tokens to be generated.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

llm = ChatNVIDIA(base_url=os.environ.get("INF_URL"), model="nvidia/nemotron-3-nano", temperature=0.1, max_tokens=1000, top_p=1.0)

result = llm.invoke("What is 2+2?")
print(result.content)

### Links and Resources

- [NVIDIA](https://docs.nvidia.com/nim/index.html)
- [nemotron-3-nano-30b-a3b](https://build.nvidia.com/nvidia/nemotron-3-nano-30b-a3b?nim=self-hosted) 

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.

<p> <center> <a href="../start_here.ipynb">Home Page</a> </center> </p>

<div>
    <span style="float: left; width: 54.5%; text-align: right;">
        <a >1</a>
        <a href="02_OpenCode_setup.ipynb">2</a>
        <a href="03_introduction_mcp.ipynb">3</a>
        <a href="04_low_level_mcp.ipynb">4</a>
        <a href="05_langraph_agent.ipynb">5</a>
        <a href="06_nemo_agent_toolkit.ipynb">6</a>
        <a href="07_challenge.ipynb">7</a>
        <a href="bonus_challenge/08_bonus_challenge.ipynb">8</a>
    </span>
    <span style="float: left; width: 45%; text-align: right;"><a href="02_OpenCode_setup.ipynb">Next Notebook</a></span>
</div>